# APIM ❤️ Microsoft Foundry

## Bring-your-own-key GitHub Copilot → Foundry lab
![flow](../../images/ghcp-byok-foundry.gif)

Let developers use **your own Azure AI Foundry model inside GitHub Copilot**, with a central **Azure API Management** gateway enforcing the things that don't come together out of the box:

1. **No model keys for callers** — developers never hold a Foundry key; APIM calls Foundry with its own managed identity.
2. **Authorization** — in EntraID mode only members of an approved security group may use the model.
3. **Central governance** — per-caller token limits + usage metering for cost/audit.
4. **No bypass** — the model can only be reached through the gateway.

This lab builds on the great work by [@angangwa](https://github.com/angangwa) **Option A** from [copilot-byok-foundry-apim](https://github.com/angangwa/copilot-byok-foundry-apim): APIM authenticates the caller, meters/limits their usage, then **drops the caller's credential and calls Foundry with its own managed identity**. Because the caller has no RBAC role on Foundry, a direct call bypassing the gateway is rejected — **identity**, not the network, makes the gateway non-bypassable - and this is the basis for Zero Trust.

### 🔑 Pick an authentication mode (`auth_type`)

Set `auth_type` in the initialization cell to choose how callers authenticate — everything else (the managed-identity swap, metering, FinOps reports and workbook) is identical either way:

- **`EntraID`** (default) — the keyless GitHub Copilot BYOK experience: developer Entra token → Microsoft Graph group check → per-user token limit → managed-identity swap, plus RFC 9728 OAuth discovery for generic clients like OpenCode. Deploys `entra-policy.xml`.
- **`ApiKey`** — callers present an **APIM subscription key** instead of an Entra token (a shared/service-key experience). No group check and no discovery; the subscription **id** and **name** map onto the same `oid`/`developerName` metric dimensions. Deploys `apikey-policy.xml`.

Both policies emit the **same** token metrics and `copilot-finops` trace, so the shared FinOps reports and Azure Monitor workbook work unchanged. In `ApiKey` mode the notebook automatically skips the Entra-only steps (group creation, Graph grant, keyless toggle, token acquisition, discovery and bypass tests).

### 🔎 OAuth discovery (Protected Resource Metadata, RFC 9728) *(EntraID mode)*

GitHub Copilot BYOK is pre-configured with the resource and tenant, but generic clients such as **[OpenCode](https://github.com/anomalyco/opencode)** don't know them ahead of time. In `EntraID` mode the gateway advertises them through **Protected Resource Metadata (PRM)** discovery so any standards-compliant client can bootstrap the OAuth flow on its own:

```mermaid
sequenceDiagram
    participant Client as OpenCode / generic client
    participant APIM as APIM gateway
    participant Entra as Entra ID (authorization server)

    Client->>APIM: POST /foundry/openai/v1/chat/completions (no token)
    APIM-->>Client: 401 + WWW-Authenticate: Bearer …, resource_metadata="…/.well-known/oauth-protected-resource"
    Client->>APIM: GET /.well-known/oauth-protected-resource (anonymous)
    APIM-->>Client: 200 { resource, authorization_servers, scopes_supported }
    Client->>Entra: interactive OAuth + PKCE (aud = resource)
    Entra-->>Client: access token
    Client->>APIM: retry with Bearer token → 200
```

The `WWW-Authenticate` hint is added in `entra-policy.xml`'s `on-error`; the metadata document is authored to satisfy OpenCode's strict rules — `resource` is `https://cognitiveservices.azure.com` (no trailing slash, matching the audience the gateway validates) and the advertised authorization server resolves its OpenID configuration at the root `…/.well-known/openid-configuration`. PRM is exercised in 🧪 Test 2 / Test 2b below and is skipped in `ApiKey` mode.

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [uv](https://docs.astral.sh/uv/) — run `uv sync` from the repo root to install dependencies
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- **Microsoft Entra privileges** *(EntraID mode only)* to create a security group and to grant an application permission (`GroupMember.Read.All`) with admin consent (e.g. *Privileged Role Administrator* / *Global Administrator*)
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...


<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription id.
- Adjust the location parameters according to your preferences and to the [product availability by Azure region](https://azure.microsoft.com/explore/global-infrastructure/products-by-region/?cdn=disable&products=cognitive-services,api-management).
- Adjust the models and versions according to the [availability by region](https://learn.microsoft.com/azure/ai-services/openai/concepts/models).

In [ ]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}" # change the name to match your naming style
resource_group_location = "swedencentral"

# ── Authentication mode ───────────────────────────────────────────────────────
# "EntraID" → developer Entra token + Graph group check + MI swap + PRM discovery
#             (the keyless GitHub Copilot BYOK experience).
# "ApiKey"  → APIM subscription key + MI swap (a shared/service key experience).
# Both modes emit the SAME copilot token metrics + trace, so the FinOps workbook
# and reports work unchanged for either choice.
auth_type = "EntraID"  # "EntraID" | "ApiKey"

aiservices_config = [{"name": "foundry1", "location": "swedencentral"}]

models_config = [{"name": "gpt-5.4", "publisher": "OpenAI", "version": "2026-03-05", "sku": "GlobalStandard", "capacity": 2000},
                # {"name": "Kimi-K2.7-Code", "publisher": "MoonshotAI", "version": "2026-04-20", "sku": "GlobalStandard", "capacity": 20}
                ]

apim_sku = 'Basicv2'  # Option A needs no private networking, so the cheaper Basic v2 tier suffices
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1"}]

inference_api_path = "foundry"        # path to the inference API in the APIM service
# exposes the OpenAI-compatible /openai/v1/* surface used by Copilot BYOK
# Use inference_api_type="PassThrough" if you plan to host multiple infrerence 'wiring API' (e.g. Chat Completions + Responses API + Anthropic's Messages API) on the same foundry
inference_api_type = "AzureOpenAIV1"
foundry_project_name = deployment_name

# The Entra security group whose members are allowed to use the gateway (created below, EntraID mode only).
allowed_group_name = f"{deployment_name}-users"

utils.print_ok(f"Notebook initialized (auth_type = {auth_type})")


<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

The following commands ensure that you have the latest version of the Azure CLI and that the Azure CLI is connected to your Azure subscription.

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

<a id='2'></a>
### 2️⃣ Create the approved Entra security group and add yourself *(EntraID mode only)*

In **EntraID** mode APIM authorizes requests by checking membership of this group via Microsoft Graph. We create it (or reuse it if it already exists) and add the currently signed-in user so you can test a successful (in-group) call.

> In **ApiKey** mode this step is skipped — the APIM subscription key is the credential, so there is no group to manage.


In [ ]:
allowed_group_id = ""  # only used in EntraID mode

if auth_type == "EntraID":
    output = utils.run(f"az ad group list --filter \"displayName eq '{allowed_group_name}'\"", f"Retrieved groups matching {allowed_group_name}", "Failed to list groups")
    if output.success and output.json_data:
        allowed_group_id = output.json_data[0]['id']
    else:
        output = utils.run(f"az ad group create --display-name {allowed_group_name} --mail-nickname {allowed_group_name}", f"Created group {allowed_group_name}", "Failed to create the group")
        if output.success and output.json_data:
            allowed_group_id = output.json_data['id']

    print(f"👉🏻 Allowed group id: {allowed_group_id}")

    # Add the currently signed-in user to the group so the in-group test succeeds.
    output = utils.run("az ad signed-in-user show", "Retrieved signed-in user", "Failed to get the signed-in user")
    if output.success and output.json_data:
        signed_in_user_id = output.json_data['id']
        add = utils.run(f"az ad group member add --group {allowed_group_id} --member-id {signed_in_user_id}", "Added user to the group", "User is likely already a member (safe to ignore)")
else:
    utils.print_info("Skipped (ApiKey mode does not use an Entra group).")


<a id='3'></a>
### 3️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declaratively define all the resources that will be deployed in the specified resource group. Change the parameters or the [main.bicep](main.bicep) directly to try different configurations.

`authType` selects the gateway policy: **EntraID** deploys [entra-policy.xml](entra-policy.xml) (token validation + Graph group check + PRM discovery) with `tenantId`/`allowedGroupId` substituted in, while **ApiKey** deploys [apikey-policy.xml](apikey-policy.xml) (subscription-key auth). Either way the Foundry backend is configured with APIM's managed-identity credentials — that is the *token swap* that lets APIM call Foundry as itself — and both policies emit the same metrics.


In [ ]:
# Create the resource group if it doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": { "value": apim_sku },
        "aiServicesConfig": { "value": aiservices_config },
        "modelsConfig": { "value": models_config },
        "apimSubscriptionsConfig": { "value": apim_subscriptions_config },
        "inferenceAPIPath": { "value": inference_api_path },
        "inferenceAPIType": { "value": inference_api_type },
        "foundryProjectName": { "value": foundry_project_name },
        "authType": { "value": auth_type },
        "tenantId": { "value": tenant_id },
        "allowedGroupId": { "value": allowed_group_id }
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Run the deployment
output = utils.run(f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed")


<a id='4'></a>
### 4️⃣ Get the deployment outputs

Retrieve the gateway URL, the APIM managed-identity principal id (needed to grant the Graph permission) and the Foundry endpoint (used later for the bypass test).

In [ ]:
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}", f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    log_analytics_id = utils.get_deployment_output(output, 'logAnalyticsWorkspaceId', 'Log Analytics Id')
    app_insights_app_id = utils.get_deployment_output(output, 'applicationInsightsAppId', 'Application Insights App Id')
    apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM Service Id')
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    apim_principal_id = utils.get_deployment_output(output, 'apimPrincipalId', 'APIM Managed Identity Principal Id')
    foundry_endpoint = utils.get_deployment_output(output, 'foundryEndpoint', 'Foundry Endpoint')
    finops_workbook_id = utils.get_deployment_output(output, 'finOpsWorkbookId', 'FinOps Workbook Id')
    prm_discovery_url = utils.get_deployment_output(output, 'prmDiscoveryUrl', 'PRM Discovery URL')
    apim_name = apim_service_id.split('/')[-1]
    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", "\""))
    # The subscription key is the caller credential in ApiKey mode.
    api_key = apim_subscriptions[0]['key']
    for subscription in apim_subscriptions:
        utils.print_info(f"Subscription Name: {subscription['name']}")


<a id='5'></a>
### 5️⃣ Grant APIM's managed identity permission to read group membership *(EntraID mode only)*

In **EntraID** mode the group check in the policy calls Microsoft Graph using APIM's managed identity, so that identity needs the **`GroupMember.Read.All`** application permission (admin-consented). This is an `appRoleAssignment` on the Microsoft Graph service principal — it can't be expressed in Bicep, so we grant it here with the Azure CLI.

> The Foundry RBAC role (*Cognitive Services User*) needed for the token swap is already assigned to APIM's managed identity by the Bicep deployment. This step only adds the separate Graph directory permission. In **ApiKey** mode it is skipped.


In [ ]:
if auth_type == "EntraID":
    graph_app_id = '00000003-0000-0000-c000-000000000000'  # Microsoft Graph
    graph_permission = 'GroupMember.Read.All'

    # 1) Resolve the Microsoft Graph service principal and the app-role id for GroupMember.Read.All
    output = utils.run(f"az ad sp show --id {graph_app_id}", "Retrieved Microsoft Graph service principal", "Failed to get the Graph service principal")
    if output.success and output.json_data:
        graph_sp_id = output.json_data['id']
        app_role_id = next((r['id'] for r in output.json_data['appRoles'] if r['value'] == graph_permission), None)
        utils.print_info(f"Graph SP object id: {graph_sp_id}")
        utils.print_info(f"'{graph_permission}' app role id: {app_role_id}")

        # 2) Assign the app role to APIM's managed identity (idempotent — a duplicate returns an error we can ignore)
        body = json.dumps({"principalId": apim_principal_id, "resourceId": graph_sp_id, "appRoleId": app_role_id})
        uri = f"https://graph.microsoft.com/v1.0/servicePrincipals/{apim_principal_id}/appRoleAssignments"
        utils.run(f"az rest --method POST --uri {uri} --headers Content-Type=application/json --body '{body}'",
            "Granted GroupMember.Read.All to APIM's managed identity", "Assignment may already exist (safe to ignore)")
else:
    utils.print_info("Skipped (ApiKey mode does not call Microsoft Graph).")


<a id='6'></a>
### 6️⃣ Make the API keyless *(EntraID mode only)*

GitHub Copilot BYOK authenticates with an Entra token only — it does not send an APIM subscription key. In **EntraID** mode we disable the subscription requirement on the API so that the developer's Entra token is the *only* credential.

> In **ApiKey** mode this step is skipped — the API keeps `subscriptionRequired = true` so the APIM subscription key is validated on every call.


In [ ]:
if auth_type == "EntraID":
    utils.run(f"az apim api update --resource-group {resource_group_name} --service-name {apim_name} --api-id inference-api --subscription-required false",
        "Disabled the subscription-key requirement (Entra-only)", "Failed to update the API")
else:
    utils.print_info("Skipped (ApiKey mode keeps the subscription-key requirement).")


<a id='7'></a>
### 7️⃣ Acquire a developer Entra token *(EntraID mode only)*

In **EntraID** mode this mirrors what the keyless Copilot BYOK client does under the hood: it asks Entra for a token whose audience is `https://cognitiveservices.azure.com` — exactly the audience the gateway validates. The token carries the signed-in user's `oid`, which the gateway uses for the group check, rate limit and metering.

> In **ApiKey** mode this step is skipped — the APIM subscription key captured in Step 4 is the credential.


In [ ]:
if auth_type == "EntraID":
    output = utils.run("az account get-access-token --resource https://cognitiveservices.azure.com", "Acquired developer Entra token", "Failed to acquire the token")
    if output.success and output.json_data:
        access_token = output.json_data['accessToken']
        utils.print_info(f"Token acquired (audience https://cognitiveservices.azure.com); expires {output.json_data.get('expiresOn', 'n/a')}")
else:
    utils.print_info("Skipped (ApiKey mode uses the subscription key, not an Entra token).")


<a id='requests'></a>
### 🧪 Test 1 — authorized caller → 200

The caller sends a chat completion. In **EntraID** mode a member of the approved group uses their Entra token (no key); the gateway validates the token, confirms group membership via Graph, meters the call, swaps in its own managed identity and forwards to Foundry. In **ApiKey** mode the caller sends the APIM subscription key instead, and the gateway meters and forwards the same way.

Tip: Use the [tracing tool](../../tools/tracing.ipynb) to track the behavior and troubleshoot the policy ([entra-policy.xml](entra-policy.xml) / [apikey-policy.xml](apikey-policy.xml)).


In [ ]:
import requests

url = f"{apim_resource_gateway_url}/{inference_api_path}/openai/v1/chat/completions"

# EntraID mode → developer token; ApiKey mode → APIM subscription key.
auth_headers = {'Authorization': 'Bearer ' + access_token} if auth_type == "EntraID" else {'api-key': api_key}

body = {
    "model": models_config[0]['name'],
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "In one sentence, what is an AI gateway?"}
    ]
}

response = requests.post(url, headers = auth_headers, json = body)
utils.print_response_code(response)

if response.status_code == 200:
    data = response.json()
    print(f"💬 {data['choices'][0]['message']['content']}")
else:
    print(response.text)


<a id='notoken'></a>
### 🧪 Test 2 — no credential → 401

Without a credential the gateway rejects the request before it ever reaches Foundry. In **EntraID** mode a missing/invalid Entra token also returns a `WWW-Authenticate` header pointing at the Protected Resource Metadata document (RFC 9728); in **ApiKey** mode a missing subscription key returns a 401 without that header.


In [ ]:
response = requests.post(url, json = body)
utils.print_response_code(response)
print('Expected 401 Unauthorized — the gateway requires a valid credential.')

# In EntraID mode the 401 carries a WWW-Authenticate header pointing at our Protected
# Resource Metadata document (RFC 9728). Discovery-capable clients use it to bootstrap
# the OAuth flow. (ApiKey mode returns a plain 401 without this header.)
www_authenticate = response.headers.get('WWW-Authenticate')
utils.print_info(f"WWW-Authenticate: {www_authenticate}")


<a id='discovery'></a>
### 🧪 Test 2b — OAuth discovery (RFC 9728) for OpenCode & other clients

GitHub Copilot BYOK is pre-configured with the resource + tenant, but generic clients like **[OpenCode](https://github.com/anomalyco/opencode)** don't know them ahead of time. They rely on **Protected Resource Metadata** discovery:

1. Client calls the inference API **without a token** → gateway returns **401 + `WWW-Authenticate`** (previous cell).
2. The header's `resource_metadata` points at `…/.well-known/oauth-protected-resource`.
3. Client `GET`s that document (anonymous) and learns the **`resource`** to request a token for and the **`authorization_servers`** to authenticate against.
4. Client runs the interactive **OAuth + PKCE** flow against Entra, then retries with the token.

> **OpenCode gotchas** — the PRM document is authored to satisfy them:
> - **Strict trailing-slash match:** `resource` is `https://cognitiveservices.azure.com` (no trailing slash), matching the audience the gateway validates.
> - **Root-path discovery:** the OpenID configuration for the listed `authorization_servers` value resolves at `<authorization_server>/.well-known/openid-configuration`.


In [ ]:
import re

if auth_type != "EntraID":
    utils.print_info("Skipped (OAuth discovery applies to EntraID mode only).")
else:
    # 1) Extract the resource_metadata URL the gateway advertised in the 401 (fall back to the deployment output).
    prm_url = prm_discovery_url
    if www_authenticate:
        m = re.search(r'resource_metadata="([^"]+)"', www_authenticate)
        if m:
            prm_url = m.group(1)
    utils.print_info(f"Discovering via: {prm_url}")

    # 2) Fetch the Protected Resource Metadata document — no token required.
    prm = requests.get(prm_url)
    utils.print_response_code(prm)
    if prm.status_code == 200:
        metadata = prm.json()
        print(json.dumps(metadata, indent=2))
        utils.print_info(f"resource: {metadata['resource']}")
        utils.print_info(f"authorization server: {metadata['authorization_servers'][0]}")
        utils.print_info(f"OpenID configuration: {metadata['authorization_servers'][0]}/.well-known/openid-configuration")


<a id='bypass'></a>
### 🧪 Test 3 — bypass attempt: developer token straight to Foundry → 401/403

This is the heart of Option A. The developer holds no RBAC role on Foundry — only APIM's managed identity does — so calling Foundry directly (skipping the gateway) is rejected on **identity** grounds, regardless of the network.

In [ ]:
if auth_type != "EntraID":
    utils.print_info("Skipped (this bypass test uses the developer Entra token, EntraID mode only).")
else:
    foundry_url = f"{foundry_endpoint}openai/v1/chat/completions"

    response = requests.post(foundry_url, headers = {'Authorization': 'Bearer ' + access_token}, json = body)
    utils.print_response_code(response)
    print('Expected 401/403 — the developer has no role on Foundry; only APIM does. The gateway is the only way in.')


<a id='finops'></a>
### 📊 FinOps reports — per-developer usage & cost

The gateway emits per-developer token metrics (dimensioned by `oid`, `developerName` and `model`) and a `copilot-finops` trace, so Copilot → Foundry spend can be attributed to the **individual developer**. Nothing is seeded — every report below is driven by **real gateway traffic** (run a few 🧪 Test 1 calls first; metrics can take a few minutes to appear).

Cost is a token × price **estimate** that discounts cached input tokens. The price table below is in USD per **1M** tokens — edit it to your negotiated rates.

> The same telemetry powers a richer, parameterized **Azure Monitor workbook** deployed by this lab (executive summary, usage concentration, model mix, live routing, chargeback and governance). See the last cell of this section for the portal link.

In [ ]:
# Editable price table (USD per 1M tokens) — change to your negotiated rates.
price_in, price_cached, price_out = 3.0, 0.3, 12.0

# Per-developer chargeback: prompt/completion/cached tokens and estimated cost, per developer × model.
chargeback_query = (
    "customMetrics "
    "| where name in ('Prompt Tokens','Completion Tokens','Prompt Cached Tokens','Total Tokens') "
    "| extend oid = tostring(customDimensions['oid']), developerName = tostring(customDimensions['developerName']), model = tostring(customDimensions['model']) "
    "| summarize Prompt = sumif(valueSum, name == 'Prompt Tokens'), Cached = sumif(valueSum, name == 'Prompt Cached Tokens'), Completion = sumif(valueSum, name == 'Completion Tokens'), Requests = sumif(valueCount, name == 'Total Tokens') by Developer = iff(developerName != '', developerName, oid), model "
    f"| extend Cost = round(((Prompt - Cached) * {price_in} + Cached * {price_cached} + Completion * {price_out}) / 1000000.0, 4) "
    "| project Developer, model, Prompt, Completion, Requests, Cost "
    "| order by Cost desc"
)

utils.run(f'az monitor app-insights query --app {app_insights_app_id} --analytics-query "{chargeback_query}"',
    "Per-developer chargeback report", "No metrics yet (they can take a few minutes to appear)")

<a id='routing'></a>
### 🔀 Live routing — served model vs requested deployment

When a router model is used, the token metric (emitted inbound, before the response) records only the *requested* name. The model actually **served** lives in the streamed response, captured by `ApiManagementGatewayLlmLog` (token + model only — no bodies, no PII). Joining it to the `copilot-finops` trace on `CorrelationId` yields per-developer served-model reporting without buffering the stream. This report reads **real gateway traffic** only.

In [ ]:
# Live routing: the router's *served* model (from ApiManagementGatewayLlmLog) vs the requested
# deployment, attributed to the developer via the copilot-finops trace on the same CorrelationId.
# This reads the resource-specific gateway tables in Log Analytics (not App Insights).
routing_query = (
    "let served = ApiManagementGatewayLlmLog | where isnotempty(ModelName) or isnotempty(DeploymentName) | project CorrelationId, ServedModel = ModelName, RequestedModel = DeploymentName, TotalTokens; "
    "let devs = ApiManagementGatewayLogs | mv-expand tr = todynamic(TraceRecords) | where tostring(tr['source']) == 'copilot-finops' | project CorrelationId, developerName = tostring(tr['metadata']['developerName']), oid = tostring(tr['metadata']['oid']); "
    "served | join kind=leftouter devs on CorrelationId | summarize Requests = count(), Tokens = sum(TotalTokens) by Developer = coalesce(developerName, oid, 'unattributed'), ServedModel, RequestedModel | order by Requests desc"
)

utils.run(f'az monitor log-analytics query --workspace {log_analytics_id} --analytics-query "{routing_query}"',
    "Live routing report (served vs requested model)", "No gateway logs yet (they can take ~15 minutes to appear)")

<a id='workbook'></a>
### 🔍 Open the FinOps workbook in the Azure portal

The lab deploys a parameterized **Azure Monitor workbook** — *Copilot FinOps - per-developer usage & cost* — that renders the reports above plus usage-concentration (Pareto/leaderboard), model mix, live routing, chargeback and rate-limit governance. It reads the same telemetry (no seeding) and lets you tweak the **price table**, **time range** and **budget** interactively.

In [ ]:
portal_url = f"https://portal.azure.com/#@{tenant_id}/resource{finops_workbook_id}"
utils.print_info(f"FinOps workbook resource: {portal_url}")
print("Open it in the Azure portal — Monitor → Workbooks → 'Copilot FinOps - per-developer usage & cost' (or the Application Insights resource → Workbooks). Adjust the Time range, Price table and Budget parameters at the top; everything recomputes live.")

<a id='vscode'></a>
### 🧩 Use it from GitHub Copilot in VS Code (BYOK)

Point VS Code Copilot at your governed gateway. Add an entry to `chatLanguageModels.json` (Command Palette → *Preferences: Open User chatLanguageModels (JSON)*), replacing the URL and model with the values printed above:

```json
{
  "name": "Foundry-via-APIM",
  "vendor": "azure",
  "models": [
    {
      "id": "gpt-5.4",
      "name": "gpt-5.4 (governed)",
      "url": "<apim_resource_gateway_url>/foundry/openai/v1/chat/completions",
      "toolCalling": true, "vision": true,
      "maxInputTokens": 128000, "maxOutputTokens": 16000
    }
  ]
}
```

In Copilot Chat → model picker → **Manage Models → Azure**, leave the **API key blank** (keyless Entra) and sign in as a user **in the approved group**. Inline grey-text completions stay GitHub-hosted — only chat/agent traffic is governed; this is expected.

<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.